# **Histograms**

## **Template Matching**

### `cv2.matchTemplate()`

> slides a small image (template) over a big image and compares how similar each position is.
> 

**`result = cv2.matchTemplate(img, template, cv2.TM_CCOEFF_NORMED)`** 

- First argument: source (big) image.
- Second argument: template (small) image.
- Third argument: method (how to compare). A common one is `cv2.TM_CCOEFF_NORMED`.
- Output `result` is a 2D NumPy array: one value for each possible template position.

### `cv2.minMaxLoc()`

> takes the result of that comparison and tells you where the best match (max or min value) is.
> 

`min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)`

- `min_val`: smallest value in `result`.
- `max_val`: largest value in `result`.
- `min_loc`: (x, y) of `min_val`.
- `max_loc`: (x, y) of `max_val`.

# Contours

## Finding contours in your image

### `cv2.findContours`

> “find shapes’ outlines” on a binary image.
> 

```python
contours, hierarchy = cv2.findContours(
    thresh,              # binary image
    cv2.RETR_EXTERNAL,   # retrieve only outer contours
    cv2.CHAIN_APPROX_SIMPLE  # compress horizontal/vertical points
)
```

### `cv2.drawContours`

> “draw those outlines” on an image.
> 

```python
cv2.drawContours(
    img,        # image to draw on
    contours,   # list of contours
    -1,         # which contour index (-1 = all)
    (0, 255, 0),# color (B,G,R)
    2           # thickness
)
```

## Convex Hull

### `cv2.convexHull`

> takes a contour (or points) and gives you the “outermost, bulged-out” boundary with no inward dents.
> 

`hull = cv2.convexHull(points, clockwise=False, returnPoints=True)` 

- `points`: contour or list/array of 2D points (often from `cv2.findContours`).
- `clockwise`: whether hull points are ordered clockwise (`True`) or counter‑clockwise (`False`, default).
- `returnPoints`:
    - `True` → returns actual hull point coordinates.
    - `False` → returns indices of hull points in the original `points` array.

## Creating Bounding boxes and circles for contours

### `cv2.boundingRect()`

> This draws a **straight, upright rectangle** around your object. It is the most common way to get a "crop" or "box" around a detected item.
> 

**`x, y, w, h = cv2.boundingRect(cnt)`** 

- cnt: from find contours

### `cv2.minEnclosingCircle()`

> This finds the **smallest possible circle** that completely covers the object.
> 

**`(center_x, center_y), radius = cv2.minEnclosingCircle(cnt)`**

## Creating Bounding rotated boxes and ellipses for contours

```python
rect = cv2.minAreaRect(cnt)
print(rect)
# ((cx, cy), (w, h), angle)

box = cv2.boxPoints(rect)            # 4 corner points as float
box = np.int0(box)                   # convert to int

cv2.drawContours(img, [box], 0, (0, 0, 255), 2)  # red rotated rectangle
```

- `(cx, cy)`: center of the rectangle (floats)
- `(w, h)`: width and height (floats)
- `angle`: rotation angle in degrees (between about -90 and 0)

```python
ellipse = cv2.fitEllipse(cnt)
print(ellipse)
# ((cx, cy), (major, minor), angle)
```

- `(cx, cy)`: center of the ellipse
- `(major, minor)`: lengths of the major and minor axes
- `angle`: rotation angle of the ellipse (degrees)

## Image Moments

### **`cv2.moments` – get centroid and more**

```python
# cnt - countours from findContours
M = cv2.moments(cnt)

# Centroid (center of mass)
cx = int(M['m10'] / M['m00'])
cy = int(M['m01'] / M['m00'])

# M['m00'] ≈ area of the contour.
# cx, cy = centroid (average x, y weighted by area).
```

### **`cv2.contourArea` – area**

```python
area = cv2.contourArea(cnt)
print("Area:", area)

# same with

M = cv2.moments(cnt)
area_from_moments = M['m00']
```

### **`cv2.arcLength` – perimeter / length**

```python
perimeter = cv2.arcLength(cnt, True)   # True = closed contour
print("Perimeter:", perimeter)
```

## Point Polygon Test

### `cv2.pointPolygonTest()` checks the relationship between a **point** and a **contour (polygon)**

It answers:

- Is the point **inside** the contour?
- Is the point **outside** the contour?
- Is the point **exactly on the contour boundary?**
- Optionally: how far is the point from the contour?

```python
result = cv2.pointPolygonTest(contour, point, measureDist)

# contour      contour from findContours()
# point        (x, y)
# measureDist  True or False
```

When `measureDist=False`:

| Return | Meaning |
| --- | --- |
| +1 | Point is inside |
| 0 | Point is on edge |
| -1 | Point is outside |

When `measureDist=True`:

| Return | Meaning |
| --- | --- |
| Positive distance | Inside contour |
| Zero | On boundary |
| Negative distance | Outside contour |

# Others

## Image Segmentation with Distance Transform and Watershed Algorithm

### `cv2.filter2D()`

> Applies a **custom convolution kernel** to an image.
> 

To create your own filters:

- Blur
- Sharpen
- Edge enhancement
- Emboss
- Motion blur
- Custom kernels

```python
dst = cv2.filter2D(src, ddepth, kernel)

src      # input image
ddepth   # output depth (-1 = same as input)
kernel   # convolution matrix
```

### `cv2.distanceTransform()`

> Distance from every foreground pixel to the nearest background pixel.
> 

To find:

- object centers
- sure foreground regions
- skeletons
- segmentation markers

```python
dist = cv2.distanceTransform(
    binary,
    cv2.DIST_L2,
    5
)

print(dist.max())    # center
```